# Iteration 7 — Multilingual Zero-Shot LLM Classification (RQ2)

## Research Questions Addressed

| RQ | Question | Target |
|----|----------|--------|
| **RQ2** | How do traditional ML models compare with transformer-based and LLM-based approaches? | Macro-F1 ≥ 0.7825 |

## Experimental Design

| Component | Detail |
|-----------|--------|
| **Model** | `joeddav/xlm-roberta-large-xnli` (560 M parameters) |
| **Method** | Zero-shot Natural Language Inference (NLI) — no fine-tuning, no few-shot examples |
| **Languages** | English, German, Swedish, Dutch |
| **Datasets** | 4 per-language JSON files from `Master Dataset 34k/By_SL_Country/` |
| **Baseline** | Iteration 0: BERT + SVM, Macro-F1 = 0.7825 (English) |
| **Why XLM-RoBERTa XNLI?** | Trained on XNLI (multilingual NLI) — handles all 4 target languages natively; zero-shot requires no labelled training data; NLI framing is well-calibrated for binary classification |
| **Process Safety Definition** | CCPS-based: equipment failure, loss of containment, emergency shutdown, fire/explosion/rupture, or release of hazardous materials from process systems |

## Outputs

- Per-language and overall classification metrics (accuracy, macro-F1, PS/NPS precision, recall, F1)
- Per-language confusion matrices (raw counts and normalised), ROC curves, PR curves, metrics bar charts — all as separate figures (8 × 6, 300 dpi, PNG + PDF)
- `iteration_7_summary.json` — structured summary with gap analysis against RQ1 target

## 1. Configuration, Path Resolution, and Constants

Establishes project root discovery, output directory creation, per-language dataset paths, the XLM-RoBERTa XNLI model identifier, descriptive candidate labels for NLI-based classification, and the hardware device for inference.

In [1]:
# =============================================================================
# IMPORTS AND CONFIGURATION — ITERATION 7
# =============================================================================
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import pipeline as hf_pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, os, json
from pathlib import Path

warnings.filterwarnings('ignore')


def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    """Search upward from start_path for directory containing 'Datasets' folder."""
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not find project root with 'Datasets' folder within {max_levels} levels. "
        f"Set THESIS_BASE_DIR environment variable or ensure Datasets folder exists."
    )


env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'master_dataset': BASE_DIR / 'Master Dataset 34k',
    'results':        BASE_DIR / 'Results' / '_iteration_7',
}
for k, p in PATHS.items():
    PATHS[k] = Path(p).resolve()
    PATHS[k].mkdir(parents=True, exist_ok=True)

RESULTS_DIR = PATHS['results']

# Per-country JSON files (from Iteration 0 outputs)
DATASET_FILES = {
    'English': PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_English_manual.json',
    'German':  PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Germany_manual.json',
    'Swedish': PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Sweden_manual.json',
    'Dutch':   PATHS['master_dataset'] / 'By_SL_Country' / 'master_df_Netherlands_manual.json',
}
DATASET_FILES = {k: v for k, v in DATASET_FILES.items() if v.exists()}
print(f"Dataset files available: {list(DATASET_FILES.keys())}")

# ─── Model ───────────────────────────────────────────────────────────────────
MODEL_NAME = "joeddav/xlm-roberta-large-xnli"

# Descriptive candidate labels — the richer the description the better the NLI match
CANDIDATE_LABELS = [
    "process safety incident involving equipment failure, leak, explosion, "
    "emergency shutdown, fire, rupture, or hazardous material release from process systems",
    "non-process safety incident involving personal injury, slip, fall, "
    "occupational hazard, vehicle accident, or administrative matter",
]
LABEL_MAP = {
    CANDIDATE_LABELS[0]: "Process Safety",
    CANDIDATE_LABELS[1]: "Non-Process Safety",
}

RQ1_TARGET = 0.85
CHECKPOINT_INTERVAL = 200  # save checkpoint every N records

# ─── Device ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    HF_DEVICE = 0
    device_label = "CUDA GPU"
elif torch.backends.mps.is_available():
    HF_DEVICE = "mps"
    device_label = "Apple MPS"
else:
    HF_DEVICE = -1
    device_label = "CPU"

print(f"[INFO] Device:      {device_label}")
print(f"[INFO] Results dir: {RESULTS_DIR}")
print(f"[INFO] Model:       {MODEL_NAME}")
print(f"[INFO] RQ1 target:  Macro F1 >= {RQ1_TARGET}")

/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
Dataset files available: ['English', 'German', 'Swedish', 'Dutch']
[INFO] Device:      Apple MPS
[INFO] Results dir: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_7
[INFO] Model:       joeddav/xlm-roberta-large-xnli
[INFO] RQ1 target:  Macro F1 >= 0.85


## 2. Data Loading — Four Language Datasets

Loads the per-language JSON files (`English`, `German`, `Swedish`, `Dutch`) from the master dataset, normalises the `CASE_TYPE` label column, and prints the class distribution per language.

In [2]:
# =============================================================================
# LOAD DATA — 4 LANGUAGES
# =============================================================================
print("[INFO] Loading per-language datasets...\n")

all_dfs = {}
for language, json_path in DATASET_FILES.items():
    df = pd.read_json(json_path)
    df['LANGUAGE'] = language

    # Normalise label: strip whitespace, title-case
    if 'CASE_TYPE' in df.columns:
        df['CASE_TYPE'] = df['CASE_TYPE'].astype(str).str.strip()

    all_dfs[language] = df

    ps_count  = (df['CASE_TYPE'] == 'Process Safety').sum()
    nps_count = (df['CASE_TYPE'] != 'Process Safety').sum()
    ps_ratio  = ps_count / len(df)
    print(f"  {language:8s}: {len(df):5,} records | PS={ps_count:4d} ({ps_ratio:.1%}) | NPS={nps_count:4d}")

# Combined master dataframe (for overall evaluation)
master_df_7 = pd.concat(all_dfs.values(), ignore_index=True)

print(f"\n  {'Total':8s}: {len(master_df_7):5,} records")
print(f"\n[INFO] Overall CASE_TYPE distribution:")
print(master_df_7['CASE_TYPE'].value_counts())
print(f"\n[INFO] Columns available: {list(master_df_7.columns[:8])} ...")

[INFO] Loading per-language datasets...

  English : 4,713 records | PS=1409 (29.9%) | NPS=3304
  German  : 14,951 records | PS=1661 (11.1%) | NPS=13290
  Swedish : 12,224 records | PS= 773 (6.3%) | NPS=11451
  Dutch   : 1,990 records | PS= 588 (29.5%) | NPS=1402

  Total   : 33,878 records

[INFO] Overall CASE_TYPE distribution:
CASE_TYPE
Safety                              24682
Process Safety                       4431
Environment                          2709
Asset and Reputation damage/loss     1322
Operational loss                      454
Information Security                  185
Physical Security                      89
Not classified                          6
Name: count, dtype: int64

[INFO] Columns available: ['CASENO', 'COMPANY', 'FUNCTIONAL_GROUP', 'FUNCTION', 'FUNCTIONAL_AREA', 'FUNCTIONAL_LOCATION', 'FUNCTIONAL_SUB_LOCATION', 'LOCATION_SID'] ...


## 3. Zero-Shot Classification Pipeline Initialisation

Loads the `joeddav/xlm-roberta-large-xnli` model via the Hugging Face `zero-shot-classification` pipeline. Includes automatic device fallback (MPS → CPU) and a smoke test to verify the model produces the expected label on a prototypical process safety incident.

In [3]:
# =============================================================================
# LOAD ZERO-SHOT CLASSIFICATION PIPELINE
# =============================================================================
print(f"[INFO] Loading model: {MODEL_NAME}")
print("[INFO] This may take 1-3 minutes on first run (downloading ~1.1 GB)...\n")

t0 = time.time()
try:
    classifier = hf_pipeline(
        "zero-shot-classification",
        model=MODEL_NAME,
        device=HF_DEVICE,
    )
    print(f"[OK] Model loaded on {classifier.device} in {time.time()-t0:.1f}s")
except Exception as e:
    print(f"[WARN] Failed to load on {HF_DEVICE}: {e}")
    print("[INFO] Retrying on CPU...")
    classifier = hf_pipeline(
        "zero-shot-classification",
        model=MODEL_NAME,
        device=-1,
    )
    print(f"[OK] Model loaded on CPU in {time.time()-t0:.1f}s")

# ─── Smoke test ──────────────────────────────────────────────────────────────
smoke_text = (
    "Title: Gas leak from pipeline flange\n"
    "Description: A high-pressure gas leak was detected at the main flange connection. "
    "Emergency shutdown was initiated and the area was evacuated."
)
result = classifier(smoke_text, candidate_labels=CANDIDATE_LABELS, multi_label=False)
predicted_label = LABEL_MAP[result['labels'][0]]
confidence = result['scores'][0]

print(f"\n[SMOKE TEST]")
print(f"  Input:     {smoke_text[:80]}...")
print(f"  Predicted: {predicted_label} (confidence: {confidence:.3f})")
print(f"  Expected:  Process Safety")
print(f"  [{'OK' if predicted_label == 'Process Safety' else 'WARN - check candidate labels'}]")

[INFO] Loading model: joeddav/xlm-roberta-large-xnli
[INFO] This may take 1-3 minutes on first run (downloading ~1.1 GB)...



Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps


[OK] Model loaded on mps in 5.0s

[SMOKE TEST]
  Input:     Title: Gas leak from pipeline flange
Description: A high-pressure gas leak was d...
  Predicted: Process Safety (confidence: 0.994)
  Expected:  Process Safety
  [OK]


## 4. Classification Helpers and Checkpoint Pipeline

Defines text preparation (`prepare_text`), single-record classification (`classify_record`), and the full checkpoint-resumable classification pipeline (`run_classification_pipeline`). Checkpoints are saved every 200 records per language so that classification can be resumed after interruption without re-processing already completed records.

In [4]:
# =============================================================================
# CLASSIFICATION HELPERS AND CHECKPOINT PIPELINE
# =============================================================================
# NOTE: Binary label mapping:
#   Positive  (1) = CASE_TYPE == 'Process Safety'
#   Negative  (0) = CASE_TYPE in {'Safety', 'Environment', 'Asset and Reputation damage/loss', ...}
# The raw CASE_TYPE field uses English labels across all language datasets.

def prepare_text(row, max_chars: int = 900) -> str:
    """
    Combine TITLE and CASE_DESCRIPTION into a single string for the NLI model.
    XLM-RoBERTa max is 512 BPE tokens (≈900 chars for mixed-language text).
    Capped at max_chars so the hypothesis (≈50 tokens) fits within the 512-token budget.
    """
    title = str(row.get('TITLE', '') or '').strip()
    desc  = str(row.get('CASE_DESCRIPTION', '') or '').strip()
    text  = f"Title: {title}\nDescription: {desc}"
    return text[:max_chars]


def classify_record(text: str, clf) -> tuple:
    """
    Run zero-shot NLI classification on a single text.
    truncation=True ensures texts beyond the model's 512-token limit are safely truncated.
    Returns: (label_str, confidence_float)
    """
    try:
        res = clf(
            text,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=False,
            truncation=True,
        )
        return LABEL_MAP[res['labels'][0]], float(res['scores'][0])
    except Exception:
        # Fallback: majority class (Non-Process Safety)
        return "Non-Process Safety", 0.0


def run_classification_pipeline(
    df: pd.DataFrame,
    language: str,
    clf,
    checkpoint_interval: int = 200
) -> tuple:
    """
    Classify all records in df for a given language with checkpoint resume.
    Returns: (predictions: list[str], confidences: list[float])
    """
    checkpoint_file = os.path.join(RESULTS_DIR, f'checkpoint_7_{language.lower()}.json')
    total = len(df)

    # ── Try to resume from checkpoint ────────────────────────────────────────
    existing = []
    if os.path.exists(checkpoint_file):
        try:
            existing = json.load(open(checkpoint_file, 'r', encoding='utf-8'))
            if len(existing) >= total:
                print(f"[INFO] {language}: Checkpoint complete ({len(existing)} records). Reusing.")
                return (
                    [r['prediction'] for r in existing],
                    [r['confidence'] for r in existing]
                )
            else:
                print(f"[INFO] {language}: Resuming from checkpoint ({len(existing)}/{total})")
        except Exception:
            print(f"[WARN] {language}: Corrupt checkpoint — starting fresh.")
            existing = []

    results = list(existing)
    start_idx = len(existing)

    ps_count = sum(1 for r in results if r['prediction'] == 'Process Safety')

    pbar = tqdm(
        range(start_idx, total),
        desc=f"{language:8s}",
        unit="rec",
        initial=start_idx,
        total=total
    )

    for idx in pbar:
        row = df.iloc[idx]
        text = prepare_text(row)
        prediction, confidence = classify_record(text, clf)

        if prediction == 'Process Safety':
            ps_count += 1

        results.append({
            'idx':          int(idx),
            'prediction':   prediction,
            'confidence':   round(confidence, 4),
            'ground_truth': str(row.get('CASE_TYPE', '')),
        })

        pbar.set_postfix({'PS': ps_count, 'conf': f"{confidence:.2f}"})

        if (idx + 1) % checkpoint_interval == 0:
            with open(checkpoint_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False)

    # Final checkpoint save
    with open(checkpoint_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False)

    predictions = [r['prediction'] for r in results]
    confidences = [r['confidence'] for r in results]
    return predictions, confidences


print("[OK] Classification helpers defined.")

[OK] Classification helpers defined.


## 5. Run Zero-Shot Classification — All Languages

Iterates over all four language datasets and runs the zero-shot NLI classification pipeline. If a checkpoint already exists with all records classified, the results are reused without re-running inference. Per-language timing and prediction counts are reported.

In [5]:
# =============================================================================
# RUN ZERO-SHOT CLASSIFICATION — ALL 4 LANGUAGES
# =============================================================================
all_language_predictions  = {}
all_language_confidences  = {}

total_start = time.time()

for language, df in all_dfs.items():
    print(f"\n{'='*65}")
    print(f"  {language}  —  {len(df):,} records")
    print(f"{'='*65}")

    lang_start = time.time()
    predictions, confidences = run_classification_pipeline(
        df, language, classifier, checkpoint_interval=CHECKPOINT_INTERVAL
    )
    elapsed = time.time() - lang_start

    all_language_predictions[language] = predictions
    all_language_confidences[language] = confidences

    ps_pred  = sum(1 for p in predictions if p == 'Process Safety')
    nps_pred = len(predictions) - ps_pred
    avg_conf = np.mean(confidences)

    print(f"\n  [DONE] {language}: PS={ps_pred}, NPS={nps_pred}, "
          f"avg_conf={avg_conf:.3f}, elapsed={elapsed/60:.1f}min")

total_elapsed = time.time() - total_start
print(f"\n{'='*65}")
print(f"[OK] All languages classified in {total_elapsed/60:.1f} minutes")
print(f"{'='*65}")


  English  —  4,713 records


English : 100%|██████████| 4713/4713 [13:31<00:00,  5.81rec/s, PS=3426, conf=0.73]



  [DONE] English: PS=3426, NPS=1287, avg_conf=0.670, elapsed=13.5min

  German  —  14,951 records


German  : 100%|██████████| 14951/14951 [44:59<00:00,  5.54rec/s, PS=13393, conf=0.74]



  [DONE] German: PS=13393, NPS=1558, avg_conf=0.652, elapsed=45.0min

  Swedish  —  12,224 records


Swedish : 100%|██████████| 12224/12224 [24:18<00:00,  8.38rec/s, PS=11154, conf=0.64]



  [DONE] Swedish: PS=11154, NPS=1070, avg_conf=0.657, elapsed=24.3min

  Dutch  —  1,990 records


Dutch   : 100%|██████████| 1990/1990 [12:56<00:00,  2.56rec/s, PS=1713, conf=0.54]



  [DONE] Dutch: PS=1713, NPS=277, avg_conf=0.648, elapsed=13.0min

[OK] All languages classified in 95.8 minutes


## 6. Evaluation Function

Computes per-class and aggregate metrics (accuracy, macro-F1, weighted-F1, PS / NPS precision, recall, F1) and the confusion matrix. Reports the gap to the RQ1 target (macro-F1 ≥ 0.85) for each evaluation.

In [6]:
# =============================================================================
# EVALUATION FUNCTION
# =============================================================================

def evaluate_classification(
    df: pd.DataFrame,
    predictions: list,
    language: str,
    print_report: bool = True
) -> dict:
    """
    Compute and optionally print full classification metrics.
    Positive class = Process Safety (label=1).
    """
    y_true = df['CASE_TYPE'].apply(lambda x: 1 if x == 'Process Safety' else 0).values
    y_pred = pd.Series(predictions).apply(lambda x: 1 if x == 'Process Safety' else 0).values

    accuracy     = accuracy_score(y_true, y_pred)
    macro_f1     = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    weighted_f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    precision_ps = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall_ps    = recall_score(   y_true, y_pred, pos_label=1, zero_division=0)
    f1_ps        = f1_score(       y_true, y_pred, pos_label=1, zero_division=0)

    precision_nps = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    recall_nps    = recall_score(   y_true, y_pred, pos_label=0, zero_division=0)
    f1_nps        = f1_score(       y_true, y_pred, pos_label=0, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])  # [[TP, FN], [FP, TN]]

    if print_report:
        print(f"\n{'='*65}")
        print(f"  RESULTS — {language}")
        print(f"{'='*65}")
        print(classification_report(
            y_true, y_pred,
            target_names=['Non-Process Safety', 'Process Safety'],
            digits=4
        ))
        status = 'ACHIEVED' if macro_f1 >= RQ1_TARGET else 'NOT YET'
        print(f"  Macro F1 : {macro_f1:.4f}   Target: {RQ1_TARGET}   Gap: {macro_f1 - RQ1_TARGET:+.4f}   [{status}]")

    return {
        'language':      language,
        'total':         int(len(df)),
        'accuracy':      float(accuracy),
        'macro_f1':      float(macro_f1),
        'weighted_f1':   float(weighted_f1),
        'precision_ps':  float(precision_ps),
        'recall_ps':     float(recall_ps),
        'f1_ps':         float(f1_ps),
        'precision_nps': float(precision_nps),
        'recall_nps':    float(recall_nps),
        'f1_nps':        float(f1_nps),
        'confusion_matrix': cm,
        'gap_to_target': float(macro_f1 - RQ1_TARGET),
    }


print("[OK] Evaluation function defined.")

[OK] Evaluation function defined.


## 7. Per-Language Evaluation

Evaluates zero-shot classification performance for each of the four languages individually, printing the full classification report and the gap to the RQ1 macro-F1 target.

In [7]:
# =============================================================================
# PER-LANGUAGE EVALUATION
# =============================================================================
all_metrics = {}

for language, df in all_dfs.items():
    predictions = all_language_predictions[language]
    metrics = evaluate_classification(df, predictions, language, print_report=True)
    all_metrics[language] = metrics


  RESULTS — English
                    precision    recall  f1-score   support

Non-Process Safety     0.8772    0.3417    0.4918      3304
    Process Safety     0.3651    0.8879    0.5175      1409

          accuracy                         0.5050      4713
         macro avg     0.6212    0.6148    0.5047      4713
      weighted avg     0.7241    0.5050    0.4995      4713

  Macro F1 : 0.5047   Target: 0.85   Gap: -0.3453   [NOT YET]

  RESULTS — German
                    precision    recall  f1-score   support

Non-Process Safety     0.9249    0.1084    0.1941     13290
    Process Safety     0.1153    0.9296    0.2051      1661

          accuracy                         0.1997     14951
         macro avg     0.5201    0.5190    0.1996     14951
      weighted avg     0.8350    0.1997    0.1953     14951

  Macro F1 : 0.1996   Target: 0.85   Gap: -0.6504   [NOT YET]

  RESULTS — Swedish
                    precision    recall  f1-score   support

Non-Process Safety     0.98

## 8. Overall Evaluation and Summary Table

Combines all per-language predictions and ground-truth labels into a single evaluation to compute overall (all-languages) metrics. Presents a summary table comparing all four languages and the combined result against the RQ1 target and the Iteration 0 baseline.

In [8]:
# =============================================================================
# OVERALL EVALUATION (ALL LANGUAGES COMBINED)
# =============================================================================
all_true_combined = []
all_pred_combined = []

for language, df in all_dfs.items():
    all_true_combined.extend(df['CASE_TYPE'].tolist())
    all_pred_combined.extend(all_language_predictions[language])

overall_df = pd.DataFrame({'CASE_TYPE': all_true_combined})
overall_metrics = evaluate_classification(
    overall_df, all_pred_combined,
    language='Overall (All 4 Languages)',
    print_report=True
)
all_metrics['Overall'] = overall_metrics

# ─── Summary table ───────────────────────────────────────────────────────────
print("\n" + "="*70)
print("  SUMMARY TABLE")
print("="*70)

summary_rows = []
for lang, m in all_metrics.items():
    summary_rows.append({
        'Language':     lang,
        'Records':      m['total'],
        'Accuracy':     f"{m['accuracy']*100:.2f}%",
        'Macro F1':     f"{m['macro_f1']*100:.2f}%",
        'PS Precision': f"{m['precision_ps']*100:.2f}%",
        'PS Recall':    f"{m['recall_ps']*100:.2f}%",
        'PS F1':        f"{m['f1_ps']*100:.2f}%",
        'NPS F1':       f"{m['f1_nps']*100:.2f}%",
        'Gap':          f"{m['gap_to_target']:+.4f}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Baseline comparison (Iteration 0, English, BERT+SVM)
it0_baseline = 0.7825
overall_f1   = all_metrics['Overall']['macro_f1']
english_f1   = all_metrics.get('English', {}).get('macro_f1', float('nan'))

print(f"\n[COMPARISON]")
print(f"  Iteration 0 baseline (EN, BERT+SVM): {it0_baseline:.4f}")
print(f"  Iteration 7 English macro F1:        {english_f1:.4f}  "
      f"({'BEATS' if english_f1 > it0_baseline else 'BELOW'} baseline)")
print(f"  Iteration 7 Overall macro F1:        {overall_f1:.4f}")
print(f"  RQ1 target:                           {RQ1_TARGET:.4f}  "
      f"({'ACHIEVED' if overall_f1 >= RQ1_TARGET else 'gap = ' + f'{RQ1_TARGET - overall_f1:.4f}'})")


  RESULTS — Overall (All 4 Languages)
                    precision    recall  f1-score   support

Non-Process Safety     0.9218    0.1312    0.2297     29447
    Process Safety     0.1382    0.9260    0.2405      4431

          accuracy                         0.2352     33878
         macro avg     0.5300    0.5286    0.2351     33878
      weighted avg     0.8193    0.2352    0.2311     33878

  Macro F1 : 0.2351   Target: 0.85   Gap: -0.6149   [NOT YET]

  SUMMARY TABLE
Language  Records Accuracy Macro F1 PS Precision PS Recall  PS F1 NPS F1     Gap
 English     4713   50.50%   50.47%       36.51%    88.79% 51.75% 49.18% -0.3453
  German    14951   19.97%   19.96%       11.53%    92.96% 20.51% 19.41% -0.6504
 Swedish    12224   14.73%   14.68%        6.74%    97.28% 12.61% 16.76% -0.7032
   Dutch     1990   40.25%   38.76%       32.46%    94.56% 48.33% 29.18% -0.4624
 Overall    33878   23.52%   23.51%       13.82%    92.60% 24.05% 22.97% -0.6149

[COMPARISON]
  Iteration 0 basel

## 9. Visualisations — Confusion Matrices, ROC Curves, PR Curves, and Metrics Bar Charts

All figures follow the standard: **Times New Roman** serif font, `figsize = (8, 6)`, `dpi = 300`, and are exported as both **PNG** and **PDF**. Each figure is saved as a separate file — no combined multi-panel images.

In [9]:
# =============================================================================
# STANDARD VISUALISATIONS — PER-LANGUAGE AND OVERALL
# =============================================================================
# Standard: Times New Roman serif, 8×6, 300 dpi, PNG + PDF dual export

def _apply_academic_rcparams():
    """Apply standard matplotlib rcParams (Times New Roman serif)."""
    matplotlib.rcParams.update({
        'font.family': 'serif',
        'font.serif':  ['Times New Roman', 'DejaVu Serif'],
        'font.size':   13,
    })


def _restore_rcparams():
    """Restore default matplotlib rcParams after plotting."""
    matplotlib.rcParams.update(matplotlib.rcParamsDefault)


def _save_academic_figure(fig, base_name):
    """Save figure as PNG (300 dpi) + PDF and close."""
    png_path = RESULTS_DIR / f'{base_name}.png'
    pdf_path = RESULTS_DIR / f'{base_name}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    print(f"  [OK] Saved: {base_name}.png + .pdf")
    plt.close(fig)


# ── Derive PS probability scores for ROC / PR curves ────────────────────────
# For zero-shot NLI: confidence is the score of the predicted label.
# If predicted == PS, ps_prob = confidence. Otherwise ps_prob = 1 - confidence.

all_language_ps_probs = {}
all_language_y_true = {}

for language, df in all_dfs.items():
    preds = all_language_predictions[language]
    confs = all_language_confidences[language]
    ps_probs = [
        conf if pred == 'Process Safety' else 1.0 - conf
        for pred, conf in zip(preds, confs)
    ]
    y_true = df['CASE_TYPE'].apply(lambda x: 1 if x == 'Process Safety' else 0).values
    all_language_ps_probs[language] = np.array(ps_probs)
    all_language_y_true[language] = y_true

# ── 1. Per-language confusion matrices (raw counts + normalised) ─────────────
for language in all_dfs.keys():
    m = all_metrics[language]
    cm = m['confusion_matrix']  # [[TP, FN], [FP, TN]] with labels=[1,0]

    # Raw counts
    _apply_academic_rcparams()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Process Safety', 'Non-Process Safety'],
        yticklabels=['Process Safety', 'Non-Process Safety'],
        cbar_kws={'label': 'Count', 'shrink': 0.8},
        linewidths=0.8, linecolor='white',
        annot_kws={'size': 18, 'fontweight': 'bold'},
        ax=ax,
    )
    ax.set_title(
        f'Confusion Matrix — {language} (XLM-RoBERTa XNLI, Zero-Shot)',
        fontsize=18, fontweight='bold', pad=14,
    )
    ax.set_ylabel('True Label', fontsize=16)
    ax.set_xlabel('Predicted Label', fontsize=16)
    ax.tick_params(labelsize=13)
    fig.tight_layout()
    _save_academic_figure(fig, f'confusion_matrix_{language.lower()}_raw')
    _restore_rcparams()

    # Normalised (% per actual class)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    _apply_academic_rcparams()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm_norm, annot=True, fmt='.1f', cmap='Blues',
        xticklabels=['Process Safety', 'Non-Process Safety'],
        yticklabels=['Process Safety', 'Non-Process Safety'],
        cbar_kws={'label': 'Percentage (%)', 'shrink': 0.8},
        linewidths=0.8, linecolor='white',
        annot_kws={'size': 18, 'fontweight': 'bold'},
        ax=ax,
    )
    ax.set_title(
        f'Normalised Confusion Matrix — {language} (XLM-RoBERTa XNLI)',
        fontsize=18, fontweight='bold', pad=14,
    )
    ax.set_ylabel('True Label', fontsize=16)
    ax.set_xlabel('Predicted Label', fontsize=16)
    ax.tick_params(labelsize=13)
    fig.tight_layout()
    _save_academic_figure(fig, f'confusion_matrix_{language.lower()}_normalised')
    _restore_rcparams()

# ── 2. Per-language ROC curves ───────────────────────────────────────────────
for language in all_dfs.keys():
    y_true = all_language_y_true[language]
    ps_probs = all_language_ps_probs[language]

    _apply_academic_rcparams()
    fpr, tpr, _ = roc_curve(y_true, ps_probs)
    auc_val = roc_auc_score(y_true, ps_probs)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='#2e86c1', lw=2, label=f'ROC Curve (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random Baseline')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=16)
    ax.set_ylabel('True Positive Rate', fontsize=16)
    ax.set_title(
        f'ROC Curve — {language} (XLM-RoBERTa XNLI, Zero-Shot)',
        fontsize=18, fontweight='bold', pad=14,
    )
    ax.legend(loc='lower right', fontsize=13)
    ax.tick_params(labelsize=13)
    fig.tight_layout()
    _save_academic_figure(fig, f'roc_curve_{language.lower()}')
    _restore_rcparams()

# ── 3. Per-language Precision-Recall curves ──────────────────────────────────
for language in all_dfs.keys():
    y_true = all_language_y_true[language]
    ps_probs = all_language_ps_probs[language]

    _apply_academic_rcparams()
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, ps_probs)
    ap = average_precision_score(y_true, ps_probs)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall_vals, precision_vals, color='#c0392b', lw=2,
            label=f'PR Curve (AP = {ap:.4f})')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall', fontsize=16)
    ax.set_ylabel('Precision', fontsize=16)
    ax.set_title(
        f'Precision-Recall Curve — {language} (XLM-RoBERTa XNLI, Zero-Shot)',
        fontsize=18, fontweight='bold', pad=14,
    )
    ax.legend(loc='upper right', fontsize=13)
    ax.tick_params(labelsize=13)
    fig.tight_layout()
    _save_academic_figure(fig, f'pr_curve_{language.lower()}')
    _restore_rcparams()

# ── 4. Per-language metrics bar charts ───────────────────────────────────────
for language in list(all_dfs.keys()) + ['Overall']:
    m = all_metrics[language]
    metrics_dict = {
        'Prec PS':  m['precision_ps'],
        'Rec PS':   m['recall_ps'],
        'F1 PS':    m['f1_ps'],
        'Prec NPS': m['precision_nps'],
        'Rec NPS':  m['recall_nps'],
        'F1 NPS':   m['f1_nps'],
        'Macro F1': m['macro_f1'],
    }

    labels = list(metrics_dict.keys())
    values = list(metrics_dict.values())
    n_ps = sum(1 for l in labels if 'PS' in l and 'NPS' not in l and 'Macro' not in l)
    n_nps = sum(1 for l in labels if 'NPS' in l)
    colors = (['#2e86c1'] * n_ps) + (['#27ae60'] * n_nps) + (['#c0392b'] * (len(labels) - n_ps - n_nps))

    _apply_academic_rcparams()
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score', fontsize=16)
    ax.set_title(
        f'Classification Metrics — {language} (XLM-RoBERTa XNLI, Zero-Shot)',
        fontsize=18, fontweight='bold', pad=14,
    )
    ax.tick_params(axis='x', rotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=13)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    fig.tight_layout()
    _save_academic_figure(fig, f'metrics_bar_{language.lower()}')
    _restore_rcparams()

# ── 5. Overall (all languages combined) confusion matrix, ROC, PR ────────────
overall_y_true = np.concatenate([all_language_y_true[l] for l in all_dfs.keys()])
overall_ps_probs = np.concatenate([all_language_ps_probs[l] for l in all_dfs.keys()])
overall_cm = all_metrics['Overall']['confusion_matrix']

# Overall confusion matrix — raw counts
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    overall_cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Process Safety', 'Non-Process Safety'],
    yticklabels=['Process Safety', 'Non-Process Safety'],
    cbar_kws={'label': 'Count', 'shrink': 0.8},
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 18, 'fontweight': 'bold'},
    ax=ax,
)
ax.set_title(
    'Confusion Matrix — Overall (XLM-RoBERTa XNLI, Zero-Shot)',
    fontsize=18, fontweight='bold', pad=14,
)
ax.set_ylabel('True Label', fontsize=16)
ax.set_xlabel('Predicted Label', fontsize=16)
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, 'confusion_matrix_overall_raw')
_restore_rcparams()

# Overall confusion matrix — normalised
cm_norm_overall = overall_cm.astype('float') / overall_cm.sum(axis=1, keepdims=True) * 100
_apply_academic_rcparams()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm_overall, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=['Process Safety', 'Non-Process Safety'],
    yticklabels=['Process Safety', 'Non-Process Safety'],
    cbar_kws={'label': 'Percentage (%)', 'shrink': 0.8},
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 18, 'fontweight': 'bold'},
    ax=ax,
)
ax.set_title(
    'Normalised Confusion Matrix — Overall (XLM-RoBERTa XNLI)',
    fontsize=18, fontweight='bold', pad=14,
)
ax.set_ylabel('True Label', fontsize=16)
ax.set_xlabel('Predicted Label', fontsize=16)
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, 'confusion_matrix_overall_normalised')
_restore_rcparams()

# Overall ROC curve
_apply_academic_rcparams()
fpr, tpr, _ = roc_curve(overall_y_true, overall_ps_probs)
auc_val = roc_auc_score(overall_y_true, overall_ps_probs)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#2e86c1', lw=2, label=f'ROC Curve (AUC = {auc_val:.4f})')
ax.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random Baseline')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=16)
ax.set_ylabel('True Positive Rate', fontsize=16)
ax.set_title(
    'ROC Curve — Overall (XLM-RoBERTa XNLI, Zero-Shot)',
    fontsize=18, fontweight='bold', pad=14,
)
ax.legend(loc='lower right', fontsize=13)
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, 'roc_curve_overall')
_restore_rcparams()

# Overall Precision-Recall curve
_apply_academic_rcparams()
precision_vals, recall_vals, _ = precision_recall_curve(overall_y_true, overall_ps_probs)
ap = average_precision_score(overall_y_true, overall_ps_probs)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_vals, precision_vals, color='#c0392b', lw=2,
        label=f'PR Curve (AP = {ap:.4f})')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall', fontsize=16)
ax.set_ylabel('Precision', fontsize=16)
ax.set_title(
    'Precision-Recall Curve — Overall (XLM-RoBERTa XNLI, Zero-Shot)',
    fontsize=18, fontweight='bold', pad=14,
)
ax.legend(loc='upper right', fontsize=13)
ax.tick_params(labelsize=13)
fig.tight_layout()
_save_academic_figure(fig, 'pr_curve_overall')
_restore_rcparams()

print('\n[OK] All per-language and overall visualisations complete.')

  [OK] Saved: confusion_matrix_english_raw.png + .pdf
  [OK] Saved: confusion_matrix_english_normalised.png + .pdf
  [OK] Saved: confusion_matrix_german_raw.png + .pdf
  [OK] Saved: confusion_matrix_german_normalised.png + .pdf
  [OK] Saved: confusion_matrix_swedish_raw.png + .pdf
  [OK] Saved: confusion_matrix_swedish_normalised.png + .pdf
  [OK] Saved: confusion_matrix_dutch_raw.png + .pdf
  [OK] Saved: confusion_matrix_dutch_normalised.png + .pdf
  [OK] Saved: roc_curve_english.png + .pdf
  [OK] Saved: roc_curve_german.png + .pdf
  [OK] Saved: roc_curve_swedish.png + .pdf
  [OK] Saved: roc_curve_dutch.png + .pdf
  [OK] Saved: pr_curve_english.png + .pdf
  [OK] Saved: pr_curve_german.png + .pdf
  [OK] Saved: pr_curve_swedish.png + .pdf
  [OK] Saved: pr_curve_dutch.png + .pdf
  [OK] Saved: metrics_bar_english.png + .pdf
  [OK] Saved: metrics_bar_german.png + .pdf
  [OK] Saved: metrics_bar_swedish.png + .pdf
  [OK] Saved: metrics_bar_dutch.png + .pdf
  [OK] Saved: metrics_bar_overall.p

## 9b. Aggregate Comparison Figures — All Languages

Generates one separate bar chart per key metric (accuracy, macro-F1, PS recall, PS precision) comparing all four languages and the combined result side by side. The RQ1 target line is overlaid on the macro-F1 chart for direct gap assessment.

In [10]:
# =============================================================================
# AGGREGATE COMPARISON FIGURES — ALL LANGUAGES
# =============================================================================

plot_langs = list(all_dfs.keys()) + ['Overall']
x = np.arange(len(plot_langs))

# ── One separate bar chart per key metric, across all languages ──────────────
aggregate_metrics = {
    'Accuracy':     [all_metrics[l]['accuracy']     for l in plot_langs],
    'Macro F1':     [all_metrics[l]['macro_f1']     for l in plot_langs],
    'PS Recall':    [all_metrics[l]['recall_ps']    for l in plot_langs],
    'PS Precision': [all_metrics[l]['precision_ps'] for l in plot_langs],
}

for metric_name, values in aggregate_metrics.items():
    _apply_academic_rcparams()
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(x, values, width=0.6, color='#2e86c1', edgecolor='white', linewidth=0.8)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    if metric_name == 'Macro F1':
        ax.axhline(y=RQ1_TARGET, color='#c0392b', linestyle='--', lw=1.5,
                   label=f'RQ1 Target (Macro F1 = {RQ1_TARGET})')
        ax.legend(fontsize=13)

    ax.set_xlabel('Language', fontsize=16)
    ax.set_ylabel(metric_name, fontsize=16)
    ax.set_title(
        f'Iteration 7 — {metric_name} by Language (XLM-RoBERTa XNLI)',
        fontsize=18, fontweight='bold', pad=14,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(plot_langs, fontsize=12)
    ax.set_ylim(0, 1.12)
    ax.tick_params(labelsize=13)
    fig.tight_layout()

    safe_name = metric_name.replace(' ', '_').lower()
    _save_academic_figure(fig, f'aggregate_{safe_name}_by_language')
    _restore_rcparams()

print('\n[OK] Aggregate comparison figures complete.')

  [OK] Saved: aggregate_accuracy_by_language.png + .pdf
  [OK] Saved: aggregate_macro_f1_by_language.png + .pdf
  [OK] Saved: aggregate_ps_recall_by_language.png + .pdf
  [OK] Saved: aggregate_ps_precision_by_language.png + .pdf

[OK] Aggregate comparison figures complete.


## 10. Final Summary Export

Saves a structured `iteration_7_summary.json` containing per-language and overall metrics, confusion matrix counts, gap analysis against the RQ1 target, and final status assessment.

In [11]:
# =============================================================================
# SAVE ITERATION 7 SUMMARY (consistent with Iteration 0 format)
# =============================================================================

def _cm_to_dict(cm):
    """Convert 2×2 numpy CM [[TP,FN],[FP,TN]] to named dict."""
    return {
        'tp': int(cm[0, 0]),
        'fn': int(cm[0, 1]),
        'fp': int(cm[1, 0]),
        'tn': int(cm[1, 1]),
    }


summary = {
    'iteration': 7,
    'model': MODEL_NAME,
    'method': 'Zero-shot NLI classification (no fine-tuning)',
    'research_question': 'RQ1 — Multilingual Process Safety Binary Classification',
    'rq1_target_macro_f1': RQ1_TARGET,
    'candidate_labels': CANDIDATE_LABELS,
    'languages': list(DATASET_FILES.keys()),
    'datasets': {},
    'gap_analysis': {},
}

for lang, m in all_metrics.items():
    cm = m['confusion_matrix']
    summary['datasets'][lang] = {
        'total':          m['total'],
        'results': {
            'accuracy':      round(m['accuracy'],      4),
            'macro_f1':      round(m['macro_f1'],      4),
            'weighted_f1':   round(m['weighted_f1'],   4),
            'precision_ps':  round(m['precision_ps'],  4),
            'recall_ps':     round(m['recall_ps'],     4),
            'f1_ps':         round(m['f1_ps'],         4),
            'precision_nps': round(m['precision_nps'], 4),
            'recall_nps':    round(m['recall_nps'],    4),
            'f1_nps':        round(m['f1_nps'],        4),
        },
        'confusion_matrix': _cm_to_dict(cm),
    }
    summary['gap_analysis'][lang] = {
        'current_macro_f1': round(m['macro_f1'], 4),
        'rq1_target':       RQ1_TARGET,
        'gap':              round(m['gap_to_target'], 4),
        'status':           'achieved' if m['macro_f1'] >= RQ1_TARGET else 'below_target',
    }

out_path = os.path.join(RESULTS_DIR, 'iteration_7_summary.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"[OK] Summary saved: {out_path}")
print()

# ─── Final status print ──────────────────────────────────────────────────────
print("=" * 65)
print("  ITERATION 7 — FINAL STATUS")
print("=" * 65)
for lang, g in summary['gap_analysis'].items():
    status = ' ACHIEVED' if g['status'] == 'achieved' else f'✗ gap={g["gap"]:+.4f}'
    print(f"  {lang:25s}: Macro F1 = {g['current_macro_f1']:.4f}  [{status}]")
print("=" * 65)

[OK] Summary saved: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/Results/_iteration_7/iteration_7_summary.json

  ITERATION 7 — FINAL STATUS
  English                  : Macro F1 = 0.5047  [✗ gap=-0.3453]
  German                   : Macro F1 = 0.1996  [✗ gap=-0.6504]
  Swedish                  : Macro F1 = 0.1468  [✗ gap=-0.7032]
  Dutch                    : Macro F1 = 0.3876  [✗ gap=-0.4624]
  Overall                  : Macro F1 = 0.2351  [✗ gap=-0.6149]
